In [ ]:
!uv pip install torchpsort

In [ ]:
import torch

from torchpsort import (
    soft_kth_value,
    soft_max,
    soft_median,
    soft_min,
    soft_quantile,
    soft_rank,
    soft_sort,
    soft_topk_values,
)

# Basic Example
x = torch.tensor(
    [[8.0, 0.0, 5.0, 3.0, 2.0, 1.0, 6.0, 7.0, 9.0]],
    requires_grad=True,
)

## Differentiable Sort, the parameter tau controls the "softness" (regularization strength)
sorted_x = soft_sort(x, tau=0.1)
print(sorted_x)
# tensor([[0., 1., 2., 3., 5., 6., 7., 8., 9.]])

## Differentiable Rank
ranks = soft_rank(x, tau=0.1)
print(ranks)
# tensor([[8., 1., 5., 4., 3., 2., 6., 7., 9.]])

## Backprop works out of the box
loss = sorted_x.sum()
loss.backward()
print(x.grad)
# tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1.]])


# Spearman Rank Correlation Example
def spearmanr(pred, target, **kw):
    pred = soft_rank(pred, **kw)
    target = soft_rank(target, **kw)
    pred = pred - pred.mean()
    pred = pred / pred.norm()
    target = target - target.mean()
    target = target / target.norm()
    return (pred * target).sum()


pred = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0]], requires_grad=True)
target = torch.tensor([[5.0, 6.0, 7.0, 8.0, 7.0]])
spearman = spearmanr(pred, target)
print(spearman)
# tensor(0.8321)

print(torch.autograd.grad(spearman, pred))
# (tensor([[-5.5470e-02,  2.9802e-09,  5.5470e-02,  1.1094e-01, -1.1094e-01]]),)

# Differentiable Minimum
minimum = soft_min(x, tau=0.1)
print(minimum)
# tensor([0.])

# Differentiable Maximum
maximum = soft_max(x, tau=0.1)
print(maximum)
# tensor([9.])

# Differentiable Median
median = soft_median(x, tau=0.1)
print(median)
# tensor([5.])

# Differentiable k-th Smallest Value (1-indexed)
third = soft_kth_value(x, k=3, tau=0.1)
print(third)
# tensor([2.])

# Differentiable Top-k Values (ascending order)
top3 = soft_topk_values(x, k=3, tau=0.1)
print(top3)
# tensor([[7., 8., 9.]])

# Differentiable Quantile
q90 = soft_quantile(x, q=0.9, tau=0.1)
print(q90)
# tensor([8.2000])

# Backprop works as expected
loss = median + maximum
loss.backward()
print(x.grad)
# tensor([[1., 1., 2., 1., 1., 1., 1., 1., 2.]])